In [1]:
from collections import defaultdict

# -----------------------------
# Search Query Dataset
# -----------------------------
queries = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit during summer",
    "best places to visit near me",
    "tourist places in kerala",
    "tourist places in goa",
    "best hotels to stay in chennai",
    "best restaurants near marina beach"
]

discount = 0.75
MAX_N = 5

# -----------------------------
# Vocabulary
# -----------------------------
vocab = {"<UNK>"}

for line in queries:
    vocab.update(line.lower().split())

# -----------------------------
# N-gram Count
# -----------------------------
grams = {n: defaultdict(int) for n in range(1, MAX_N + 1)}
hist = {n: defaultdict(int) for n in range(2, MAX_N + 1)}

for line in queries:

    words = line.lower().split()

    for n in range(1, MAX_N + 1):

        if len(words) < n:
            continue

        for i in range(len(words) - n + 1):

            ngram = tuple(words[i:i+n])
            grams[n][ngram] += 1

            if n > 1:
                hist[n][ngram[:-1]] += 1

# -----------------------------
# Followers
# -----------------------------
next_words = {n: defaultdict(set) for n in range(2, MAX_N + 1)}

for n in range(2, MAX_N + 1):

    for ngram in grams[n]:

        next_words[n][ngram[:-1]].add(ngram[-1])

# -----------------------------
# Kneser-Ney Probability
# -----------------------------
def kn_prob(context, word):

    if word not in vocab:
        word = "<UNK>"

    order = len(context) + 1

    if order == 1:

        total = sum(grams[1].values())

        return (grams[1].get((word,), 0) + 1) / (total + len(vocab))

    context = tuple(context)
    ngram = context + (word,)

    context_count = hist[order].get(context, 0)

    if context_count == 0:
        return kn_prob(context[1:], word)

    count = grams[order].get(ngram, 0)

    first = max(count - discount, 0) / context_count

    lam = (discount * len(next_words[order][context])) / context_count

    backoff = kn_prob(context[1:], word)

    return first + lam * backoff

# -----------------------------
# Auto Complete
# -----------------------------
def auto_complete(text, k=5):

    words = text.lower().split()

    context = words[-4:] if len(words) >= 4 else words

    result = []

    for word in vocab:

        if word == "<UNK>":
            continue

        prob = kn_prob(context, word)

        result.append((prob, word))

    result.sort(reverse=True)

    print("\nSuggestions for:", text)

    for prob, word in result[:k]:
        print(f"{text} {word}  -> {prob:.4f}")

# -----------------------------
# Test
# -----------------------------
auto_complete("best places to visit")
auto_complete("places to visit in")
auto_complete("best hotels to stay")


Suggestions for: best places to visit
best places to visit in  -> 0.6523
best places to visit near  -> 0.1333
best places to visit during  -> 0.1317
best places to visit places  -> 0.0111
best places to visit best  -> 0.0111

Suggestions for: places to visit in
places to visit in chennai  -> 0.4066
places to visit in india  -> 0.3182
places to visit in kerala  -> 0.0291
places to visit in goa  -> 0.0291
places to visit in places  -> 0.0281

Suggestions for: best hotels to stay
best hotels to stay in  -> 0.7137
best hotels to stay places  -> 0.0352
best hotels to stay best  -> 0.0352
best hotels to stay to  -> 0.0301
best hotels to stay visit  -> 0.0251
